# AI-Powered Text Paraphrasing Tool Using Transformer-Based Models

This AI-powered text paraphrasing model is a Python-based system that rewrites input text into a new version while keeping the original meaning intact. It uses pretrained transformer models from Hugging Face Transformers to generate paraphrases that are clearer, more natural, and less repetitive. The model also applies basic language checks using spaCy to improve grammar and sentence structure, ensuring the output is readable and fluent.

## Objectives:
* Rewrite input text while preserving its original meaning
* Improve clarity and readability of sentences
* Generate more original phrasing to reduce duplication or repetition
* Utilize transformer-based models through Hugging Face Transformers for accurate paraphrasing
* Perform basic grammar and structure checks using spaCy
* Evaluate output quality using metrics like BLEU and ROUGE
* Provide a simple, script-based solution for text paraphrasing without requiring a graphical interface



In [1]:
!pip install transformers torch sentencepiece spacy nltk rouge-score
!python -m spacy download en_core_web_sm

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=eb6ca9c5f44149b05aaf45b5ec5ff2ef49d0ffeb907e51da6e5ac4cbcd62b532
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 110.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## Import Libraries

In [2]:
# Import Libraries
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch
import spacy
from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
from rouge_score import rouge_scorer

In [3]:
# Load spaCy model for grammar checks
nlp = spacy.load("en_core_web_sm")

## Load Pre-Trained Model

*   Vamsi/T5_Paraphrase_Paws

T5 is a transformer model that converts every NLP task into a text-to-text problem. It takes a text input and generates a text output, which makes it suitable for tasks like paraphrasing, translation, and summarization. Because it uses an encoder–decoder architecture, it can both understand and rewrite text effectively.

This is a fine-tuned T5 model designed specifically for paraphrasing tasks. It rewrites sentences while preserving their original meaning, using training data like PAWS that focuses on structural variations. The model is reliable for generating grammatically correct and semantically accurate paraphrases, but it tends to produce conservative outputs that are often very similar to the input text.

In [4]:
# Load T5 model and tokenizer
model_name = "Vamsi/T5_Paraphrase_Paws"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

In [5]:
def paraphrase_text(text):
    input_text = "paraphrase: " + text

# Encoding
    encoding = tokenizer(
        input_text,
        padding="longest",
        return_tensors="pt",
        truncation=True
    )

    input_ids = encoding["input_ids"]
    attention_mask = encoding["attention_mask"]

    outputs = model.generate(
    input_ids=input_ids,
    attention_mask=attention_mask,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    temperature=0.9,
    max_length=256,
    num_return_sequences=5,
    repetition_penalty=2.5
)

# Decoding
    paraphrases = [
    tokenizer.decode(o, skip_special_tokens=True)
    for o in outputs
]

# Remove identical to input
    paraphrases = list(set([p for p in paraphrases if p.lower().strip() != text.lower().strip()]))

    return paraphrases

## Grammar Check

In [6]:
def check_grammar(text):
    doc = nlp(text)
    has_subject = any(token.dep_ in ("nsubj", "nsubjpass") for token in doc)
    has_verb = any(token.pos_ in ("VERB", "AUX") for token in doc)

    return {
        "sentences": len(list(doc.sents)),
        "tokens": len(doc),
        "is_valid": has_subject and has_verb
    }

## Model Evaluation

In [7]:
def evaluate(original, paraphrased):

    chencherry = SmoothingFunction()

    bleu = sentence_bleu(
        [original.split()],
        paraphrased.split(),
        weights=(0.5, 0.5, 0, 0),
        smoothing_function=chencherry.method1
    )

    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    rouge = scorer.score(original, paraphrased)

    return {
        "BLEU (Similarity)": bleu,
        "ROUGE-1": rouge['rouge1'].fmeasure,
        "ROUGE-L": rouge['rougeL'].fmeasure
    }

In [9]:
if __name__ == "__main__":
    print("\nNihil's AI Paraphrasing Tool")
    print("------------------------------")

    user_input = input("\nEnter Your Text:\n")

    # Paraphrase
    paraphrased_text_list = paraphrase_text(user_input)

    # Choose one paraphrased text for grammar check and evaluation
    if paraphrased_text_list:
        selected_paraphrased_text = paraphrased_text_list[0]
    else:
        selected_paraphrased_text = user_input
        print("Warning: No paraphrases generated!. Using original text for grammar check and evaluation.")

    # Grammar check
    grammar_info = check_grammar(selected_paraphrased_text)

    # Evaluation
    scores = evaluate(user_input, selected_paraphrased_text)

    # Output results
    print("\nOriginal Text:\n", user_input)
    print("\nParaphrased Text:\n", selected_paraphrased_text)
    print("\nAll Paraphrases Generated:\n", paraphrased_text_list)

    print("\nGrammar Check:", grammar_info)
    print("\nEvaluation Scores:", scores)


Nihil's AI Paraphrasing Tool
------------------------------

Enter Your Text:
Mohanlal Viswanathan born 21 May 1960, known mononymously as Mohanlal, is an Indian actor, filmmaker, producer and playback singer who predominantly works in Malayalam cinema and has also occasionally appeared in Tamil, Hindi, Telugu and Kannada films.Mohanlal has a prolific career spanning over four decades, during which he has acted in more than 400 films. The Government of India honoured him with Padma Shri in 2001 and Padma Bhushan in 2019, India's fourth and third highest civilian honours, for his contributions to Indian cinema. In 2009, he became the first actor in India to be awarded the honorary rank of lieutenant colonel in the Territorial Army.

Original Text:
 Mohanlal Viswanathan born 21 May 1960, known mononymously as Mohanlal, is an Indian actor, filmmaker, producer and playback singer who predominantly works in Malayalam cinema and has also occasionally appeared in Tamil, Hindi, Telugu and Kan